# Strawberry Data Audit

Use this notebook before changing model configs. It confirms split health, label ranges, real sensor mapping status, and sequence counts.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

LAB_DIR = Path.cwd()
if LAB_DIR.name != 'strawberry':
    LAB_DIR = Path('notebooks/strawberry').resolve()
sys.path.insert(0, str(LAB_DIR))
import lab_utils as lab


In [ ]:
labels = lab.load_all_labels()
print(labels.shape)
columns = ['split', 'fruit_id', 'timestamp', 'rul_hours', 'temperature_c', 'humidity_pct']
columns += [c for c in ['environment_source', 'sensor_status'] if c in labels.columns]
labels[columns].head()


In [ ]:
summary = labels.groupby(['split', 'fruit_id']).agg(
    frames=('rul_hours', 'size'),
    rul_min=('rul_hours', 'min'),
    rul_mean=('rul_hours', 'mean'),
    rul_median=('rul_hours', 'median'),
    rul_max=('rul_hours', 'max'),
    pct_zero=('rul_hours', lambda s: (s == 0).mean()),
    temp_missing=('temperature_c', lambda s: s.isna().mean()),
    hum_missing=('humidity_pct', lambda s: s.isna().mean()),
).reset_index()
summary


In [ ]:
if 'sensor_status' in labels.columns:
    display(labels['sensor_status'].value_counts(dropna=False))
if 'environment_source' in labels.columns:
    display(labels['environment_source'].value_counts(dropna=False))


In [ ]:
labels[['rul_hours', 'temperature_c', 'humidity_pct']].corr(numeric_only=True)


In [ ]:
counts = lab.sequence_counts([3, 5, 8, 10, 12])
counts.groupby(['split', 'seq_len'])['sequences'].sum().unstack('seq_len')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels['rul_hours'].hist(ax=axes[0], bins=40)
axes[0].set_title('RUL distribution')
labels['temperature_c'].hist(ax=axes[1], bins=20)
axes[1].set_title('Temperature')
labels['humidity_pct'].hist(ax=axes[2], bins=20)
axes[2].set_title('Humidity')
plt.tight_layout()


## Notes

Temperature and humidity must come from real sensor mapping. If `sensor_status` is missing or not `matched`, do not treat those values as measured evidence; rerun preprocessing with a real sensor CSV or keep missing-sensor flags in the model pipeline.
